# Homework 1.1: Can an AI Rediscover Kepler's Orbit of Mars?
## Mathematical Foundations of Generative AI
### Project Goal
The goal of this project was to use a large language model (LLM) as a scientific collaborator rather than as an answer generator. The project built a synthetic version of the observational problem faced by Kepler: generate noisy measurements of Mars from Uraniborg, reconstruct heliocentric Mars positions without inserting the hidden Mars orbital parameters into the reconstruction algorithm, fit competing geometric models, and then judge whether the data support a circle or an ellipse. The hidden Mars orbit is used only in a separate post-hoc verification notebook after the reconstruction has already been produced. That verification does not feed into the fitted models, residuals, or noise comparison.



The workflow was:



1. Generate synthetic Tycho-like observations of Mars from 1582 to 1600.

2. Independently verify the noiseless orbital model using conservation of energy and angular momentum.

3. Plot Mars in the apparent sky using right ascension/declination and altitude/azimuth.

4. Convert angular measurements to geometric lines of sight and reconstruct heliocentric Mars positions.

5. Plot the reconstructed Mars positions together with the Sun and Earth's orbit.

5.1. Perform a separate post-hoc check by comparing the already reconstructed 1 arcminute points with the hidden generating orbit. This check is for validation only and is not used by the later fitting steps.

6. Fit a Sun-centered circle, an arbitrary-center circle, and a focus-constrained ellipse with the Sun at one focus.

7. Repeat the analysis for 1, 2, and 5 arcminutes of observational noise.

8. Ask the LLM to interpret the numerical evidence and critique its reasoning.

9. Discuss the connection between Kepler's work and modern model fitting from data.



---
# 1. Synthetic Observation Table
Three synthetic observation tables were generated, one for each noise level:



- [1 arcminute observations](data/synthetic_mars_1arcmin.csv)

- [2 arcminute observations](data/synthetic_mars_2arcmin.csv)

- [5 arcminute observations](data/synthetic_mars_5arcmin.csv)



Each file contains 166 accepted observations. The columns are:



- `datetime_utc`: observation date and time

- `ra_deg`: right ascension in degrees

- `dec_deg`: declination in degrees

- `alt_deg`: altitude above the Uraniborg horizon in degrees

- `az_deg`: azimuth measured clockwise from north in degrees

- `noise_arcmin`: angular noise level used for that table



The observations were generated every 10 days at 22:00 UTC between 1582 and 1600, but an observation was kept only when Mars was at least 10 degrees above the horizon and the Sun was at least 6 degrees below the horizon. This prevents the synthetic observer from recording Mars when it would be below the horizon or in bright daylight.



A sample from the 1 arcminute table is shown below. The complete table is included in the submitted CSV file.



| Date (UTC) | RA (deg) | Dec (deg) | Altitude (deg) | Azimuth (deg) | Noise |
| --- | ---: | ---: | ---: | ---: | ---: |
| 1582-10-18 22:00 | 113.1434 | 21.7463 | 10.6741 | 66.7871 | 1' |
| 1582-10-28 22:00 | 116.7729 | 21.1738 | 13.4672 | 72.0128 | 1' |
| 1582-11-07 22:00 | 119.5004 | 20.6809 | 16.9230 | 77.9334 | 1' |
| 1582-11-17 22:00 | 121.1073 | 20.3617 | 21.2341 | 84.7275 | 1' |
| 1582-11-27 22:00 | 121.4645 | 20.2909 | 26.4962 | 92.5909 | 1' |
| 1582-12-07 22:00 | 120.3862 | 20.5386 | 32.7699 | 101.9267 | 1' |
| 1582-12-17 22:00 | 117.8709 | 20.9645 | 39.7199 | 113.4277 | 1' |
| 1582-12-27 22:00 | 114.1580 | 21.5882 | 46.7929 | 127.8897 | 1' |
| 1583-01-06 22:00 | 109.9395 | 22.1667 | 52.7118 | 146.0109 | 1' |
| 1583-01-16 22:00 | 106.0045 | 22.6162 | 56.2296 | 167.2120 | 1' |
| 1583-01-26 22:00 | 103.1156 | 22.9017 | 56.7807 | 188.5722 | 1' |
| 1583-02-05 22:00 | 101.5174 | 22.9913 | 54.8962 | 207.1125 | 1' |



The full 1 arcminute data range was approximately:



- RA: 0.89 to 355.91 degrees

- Declination: -23.14 to 23.44 degrees

- Altitude: 10.25 to 57.48 degrees

- Azimuth: 66.03 to 294.84 degrees



The 2 and 5 arcminute tables contain the same underlying observations with independently perturbed angular directions.



---
# 2. Hidden Orbital Model Used to Generate the Data
The synthetic model is intentionally simplified. The Sun is fixed at the origin, and Earth and Mars move on fixed Kepler ellipses in a common ecliptic plane. Planetary perturbations, precession, and a full historical ephemeris are not included. The purpose is to create a controlled data-fitting problem rather than a high-precision astronomical simulator.



The observer is located at Uraniborg using the coordinates



- latitude: 55 degrees 54 minutes 28 seconds N

- longitude: 12 degrees 41 minutes 48 seconds E



The hidden orbital parameters used only in data generation were:



| Planet | Semi-major axis a (AU) | Eccentricity e | Initial mean anomaly M0 | Perihelion direction |
| --- | ---: | ---: | ---: | ---: |
| Earth | 1.00000011 | 0.01671022 | 357.51716 deg | 102.93735 deg |
| Mars | 1.52366231 | 0.09341233 | 19.41248 deg | 336.04084 deg |



With distance measured in AU and time measured in days, the solar gravitational parameter was taken as



$$

\mu = \frac{4\pi^2}{(365.256363004)^2}

$$



For a planet with semi-major axis $a$, the mean angular rate is



$$

n = \sqrt{\frac{\mu}{a^3}}

$$



and the mean anomaly is



$$

M(t) = M_0 + nt

$$



The eccentric anomaly $E$ is obtained by solving Kepler's equation



$$

E - e\sin(E) = M

$$



Newton's method was used to solve this nonlinear equation. Once $E$ is known, the position in the orbital plane is



$$

x = a(\cos E - e)

$$



$$

y = a\sqrt{1-e^2}\sin E

$$



The velocity is obtained by differentiating Kepler's equation. Since



$$

\frac{dE}{dt} = \frac{n}{1-e\cos E}

$$



the velocity components are



$$

v_x = -a\sin E\,\frac{dE}{dt}

$$



$$

v_y = a\sqrt{1-e^2}\cos E\,\frac{dE}{dt}

$$



The orbital plane is rotated by the chosen perihelion direction.



## Observer geometry
Earth's axial tilt was taken to be 23.43928 degrees. The heliocentric planet positions were converted from ecliptic coordinates to equatorial coordinates. Earth's daily rotation was represented using an approximate Greenwich mean sidereal time calculation. Uraniborg's longitude shifts the sidereal angle to the local angle at the observatory, while its latitude determines the local east, north, and up basis.



The observer's heliocentric position is



$$

\mathbf{r}_{\mathrm{obs}} = \mathbf{r}_{E} + \mathbf{r}_{\mathrm{site}}

$$



The direction from Uraniborg to Mars is



$$

\boldsymbol{\rho} = \mathbf{r}_{M} - \mathbf{r}_{\mathrm{obs}}

$$



After normalization, this unit vector is converted to right ascension/declination and altitude/azimuth.



## Observational noise
Noise was added directly to the unit line-of-sight vector in its local tangent plane. This is more physically meaningful than independently adding random values to RA and declination because it represents a small random pointing error on the celestial sphere. The quoted arcminute value is the scale used for the tangent-plane Gaussian perturbation; an individual realized angular offset is random and need not equal that value exactly. Three angular noise scales were tested:



- 1 arcminute

- 2 arcminutes

- 5 arcminutes



The synthetic-data generation code is contained in [synthetic_data_generation(step_1).ipynb](codes/synthetic_data_generation(step_1).ipynb).



---
# 3. Verification of the Noiseless Orbital Simulation
The orbital code was checked independently rather than accepted because it was generated with LLM assistance. Two conserved quantities were tested: specific orbital energy and specific angular momentum.



For a Kepler orbit, the specific orbital energy is



$$

\mathcal{E} = \frac{1}{2}\lVert\mathbf{v}\rVert^2 - \frac{\mu}{\lVert\mathbf{r}\rVert}

$$



and the exact theoretical value is



$$

\mathcal{E}_{\mathrm{theory}} = -\frac{\mu}{2a}

$$



The specific angular momentum is



$$

h = \lVert\mathbf{r} \times \mathbf{v}\rVert

$$



with theoretical value



$$

h_{\mathrm{theory}} = \sqrt{\mu a(1-e^2)}

$$



The independent verification script evaluated the Earth and Mars orbits from 1582 through 1600 and compared the computed invariants with these closed-form values.



## Verification summary
| Planet | Period (days) | Perihelion (AU) | Aphelion (AU) | Max relative energy error | Energy relative variation | Max relative angular momentum error | Angular momentum relative variation |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| Earth | 365.2564 | 0.983290 | 1.016710 | 1.28e-15 | 2.38e-15 | 6.05e-16 | 1.01e-15 |
| Mars | 686.9591 | 1.381333 | 1.665991 | 1.40e-15 | 2.51e-15 | 8.21e-16 | 1.15e-15 |



These errors are at the scale of floating-point roundoff. Therefore the position and velocity formulas are internally consistent with the theoretical Kepler invariants.



One limitation of this check is important: the orbit is generated analytically from Kepler's equation rather than integrated forward with a numerical ODE solver. Therefore near-perfect conservation is expected. The verification confirms that the formulas used for position and velocity are mutually consistent; it is not a test of long-time numerical integration stability.



### Earth energy conservation
![Earth energy verification](result%20images/earth_energy_verification.png)



### Earth angular momentum conservation
![Earth angular momentum verification](result%20images/earth_angular_momentum_verification.png)



### Mars energy conservation
![Mars energy verification](result%20images/mars_energy_verification.png)



### Mars angular momentum conservation
![Mars angular momentum verification](result%20images/mars_angular_momentum_verification.png)



The complete numerical verification table is stored in [orbit_verification_summary.csv](data/orbit_verification_summary.csv).



---
# 4. Apparent Position of Mars in Tycho's Sky
The synthetic observations were plotted in two coordinate systems.



Right ascension and declination describe the direction of Mars on the celestial sphere. Altitude and azimuth describe what an observer at Uraniborg would see relative to the local horizon. The latter coordinates make the effect of Earth's rotation and Uraniborg's geographic location especially clear.



## Right ascension and declination
### 1 arcminute noise
![Mars RA-Dec 1 arcmin](result%20images/mars_radec_1arcmin.png)



### 2 arcminutes noise
![Mars RA-Dec 2 arcmin](result%20images/mars_radec_2arcmin.png)



### 5 arcminutes noise
![Mars RA-Dec 5 arcmin](result%20images/mars_radec_5arcmin.png)



The apparent path is not expected to look like a simple ellipse because the observer is moving with Earth. The plotted direction depends on the relative vector between Mars and the observer, not simply on Mars's heliocentric coordinates. Apparent reversals and loops are therefore natural in the sky coordinates.



## Altitude and azimuth
### 1 arcminute noise
![Mars altitude-azimuth 1 arcmin](result%20images/mars_altaz_1arcmin.png)



### 2 arcminutes noise
![Mars altitude-azimuth 2 arcmin](result%20images/mars_altaz_2arcmin.png)



### 5 arcminutes noise
![Mars altitude-azimuth 5 arcmin](result%20images/mars_altaz_5arcmin.png)



The lower boundary near 10 degrees altitude is a direct consequence of the visibility condition used during data generation.



The three noise levels look visually similar because even 5 arcminutes is only about 0.083 degrees, much smaller than the overall angular scale of the sky plots.



---
# 5. Reconstruction of Heliocentric Mars Positions
A single RA/declination observation provides a direction but not a distance. If $\mathbf{p}_i$ is the heliocentric observer position and $\mathbf{u}_i$ is the observed unit direction, then Mars must lie somewhere on the ray



$$

\mathbf{r} = \mathbf{p}_i + \rho_i\mathbf{u}_i

$$



where the range $\rho_i$ is unknown.



## Corrected multi-ray reconstruction
The final reconstruction algorithm did not insert the hidden Mars semi-major axis, eccentricity, orientation, or true period. It used:

- the noisy RA/declination observations,
- Earth's known orbit,
- Uraniborg's location,
- Earth's rotation,
- an additional modeling assumption that Mars has approximately periodic heliocentric motion.

The method searched candidate repetition periods between 500 and 900 days. This interval is itself a broad prior rather than something forced by a single line-of-sight observation, so it should be regarded as part of the reconstruction model. Observations separated by approximately one or more candidate periods were grouped under the assumption that Mars should return near the same heliocentric orbital phase. At least three lines of sight were required for each reconstructed point.

For a group with observer positions $\mathbf{p}_i$ and unit directions $\mathbf{u}_i$, define

$$
P_i = I-\mathbf{u}_i\mathbf{u}_i^T.
$$

The reconstructed common point $\mathbf{x}$ minimizes the total squared perpendicular distance to the corresponding lines:

$$
\min_{\mathbf{x}} \sum_i \left\|P_i(\mathbf{x}-\mathbf{p}_i)\right\|^2.
$$

Setting the gradient equal to zero gives the normal equations

$$
A\mathbf{x}=\mathbf{b},
$$

where

$$
A=\sum_i P_i
$$

and

$$
\mathbf{b}=\sum_i P_i\mathbf{p}_i.
$$

The least-squares derivation is naturally written for infinite lines. The implementation then enforces the physical ray constraint by computing

$$
\rho_i = \mathbf{u}_i^T(\mathbf{x}-\mathbf{p}_i)
$$

and rejecting any reconstructed bundle for which one or more recovered ranges satisfy $\rho_i \le 0$. Therefore every accepted reconstructed point lies in front of all contributing observers.

The code also protects against poorly conditioned triangulation in two separate ways. First, it requires a minimum heliocentric observer baseline of 0.30 AU. Second, it computes the condition number of the normal matrix $A$ and rejects a bundle when $\operatorname{cond}(A)>10^{10}$. These checks are important because a small ray-intersection residual alone does not guarantee that the recovered distance is stable.

The reconstructed points should be expressed in one consistent Cartesian frame. In particular, the near-zero reported $z$ values are interpreted in the ecliptic frame; this frame must also be used consistently when fitting the planar circle and ellipse models.

The best inferred repetition period was 689.25 days for all three noise levels. Only after this period had been inferred was it compared with the hidden generating value 686.959 days. The difference is about 2.29 days, or approximately 0.33 percent.

## Reconstruction statistics
| Noise | Reconstructed points | Inferred period (days) | Minimum radius (AU) | Median radius (AU) | Maximum radius (AU) | Median ray RMS (AU) | Median absolute z (AU) |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 1' | 26 | 689.25 | 1.4268 | 1.5387 | 1.6072 | 0.01235 | 0.000134 |
| 2' | 26 | 689.25 | 1.4262 | 1.5380 | 1.6071 | 0.01236 | 0.000267 |
| 5' | 26 | 689.25 | 1.4245 | 1.5359 | 1.6070 | 0.01242 | 0.000667 |



After Step 4 was completed, the hidden model was used only as an external verification check. It places Mars between approximately 1.381 and 1.666 AU from the Sun, while the reconstructed radii lie in a similar range. This agreement supports the physical plausibility of the reconstruction, but these hidden values are not used in the period search, ray-bundle selection, reconstructed coordinates, model fits, or residual calculations. The reported $z$ coordinates also remain close to zero in the ecliptic frame, consistent with the planar hidden model.



### Period search, 1 arcminute
![Period search 1 arcmin](result%20images/period_search_1arcmin.png)



### Reconstructed Mars positions, 1 arcminute
![Mars reconstruction 1 arcmin](result%20images/mars_reconstruction_1arcmin.png)



### Reconstructed Mars positions, 2 arcminutes
![Mars reconstruction 2 arcmin](result%20images/mars_reconstruction_2arcmin.png)



### Reconstructed Mars positions, 5 arcminutes
![Mars reconstruction 5 arcmin](result%20images/mars_reconstruction_5arcmin.png)



## Post-hoc verification against the hidden orbit
Step 5-1 is a validation check, not part of the inference pipeline. It explicitly contains the hidden Mars parameters and uses them to draw the true generating orbit, but it first reads the already completed Step 4 file `mars_reconstructed_1arcmin.csv`. It does not modify the reconstructed coordinates or write replacement reconstruction files.

The computational dependency is therefore

```text
Step 4 reconstruction
    -> mars_reconstructed_1arcmin.csv
    -> mars_reconstructed_2arcmin.csv
    -> mars_reconstructed_5arcmin.csv

Step 5-1 verification
    -> reads mars_reconstructed_1arcmin.csv
    -> compares it with the hidden generating orbit
    -> produces a verification plot only

Step 6 model fitting
    -> independently reads the Step 4 mars_reconstructed_*.csv files
    -> produces model_fit_summary.csv

Step 7 noise comparison
    -> reads model_fit_summary.csv
```

Thus the hidden Mars parameters in Step 5-1 do not enter the period estimate, reconstructed positions, fitted circles, fitted ellipse, RMSE values, or the 1/2/5 arcminute comparison. The role of Step 5-1 is simply to check that the blind reconstruction obtained in Step 4 is consistent with the known synthetic ground truth.

The reconstructed 1 arcminute points follow the same approximate scale, orientation, and curvature as the hidden generating ellipse, providing an additional sanity check on the final reconstruction.

![Reconstruction compared with hidden orbit](result%20images/reconstruction_vs_hidden_orbit_1arcmin.png)

An earlier pairwise reconstruction was numerically unstable because nearly repeated observer geometry gives weak parallax information. The final Step 4 method addresses that problem internally by using at least three rays, requiring a minimum observer baseline, rejecting ill-conditioned normal matrices, and enforcing positive ray ranges. Step 5-1 then serves only as an independent confirmation that the corrected reconstruction is physically consistent with the synthetic orbit.


---
# 6. Circle and Ellipse Fits with Residual Errors
Three models were fitted to each reconstructed data set.



## Model 1: Sun-centered circle
The model is



$$

x^2 + y^2 = R^2

$$



Only the radius $R$ is fitted.



## Model 2: Free-center circle
The model is



$$

(x-h)^2 + (y-k)^2 = R^2

$$



The center $(h,k)$ and radius $R$ are fitted.



## Model 3: Focus-constrained ellipse with the Sun at one focus
The assignment asks for an ellipse. Here I used the physically motivated three-parameter subfamily in which the Sun is constrained to be one focus. This is still an ellipse, and it has the same number of fitted parameters as the free-center circle, but it uses more physical structure than a completely general five-parameter planar ellipse. A fully model-agnostic extension would also fit a general ellipse and then test whether one of its foci lies near the Sun.

The polar form is

$$

r(\theta) = \frac{p}{1+e\cos(\theta-\omega)}

$$



The fitted parameters are the semi-latus rectum $p$, eccentricity $e$, and orientation $\omega$. The semi-major axis is then



$$

a = \frac{p}{1-e^2}

$$



To compare all three models fairly, the final residual for each reconstructed point was the Euclidean distance to the nearest point on the fitted curve:



$$

d_i = \min_{\mathbf{q}\in C} \lVert \mathbf{x}_i - \mathbf{q} \rVert

$$



The reported RMSE values therefore use the same geometric error definition for all models. The free-center circle and the focus-constrained ellipse each have three fitted parameters, so their raw RMSE values are directly comparable in parameter count; the Sun-centered circle has only one fitted parameter:



$$

\mathrm{RMSE} = \sqrt{\frac{1}{N}\sum_{i=1}^{N} d_i^2}

$$



## Residual comparison
| Noise | Model | RMSE (AU) | MAE (AU) | Median error (AU) | Maximum error (AU) |
| ---: | --- | ---: | ---: | ---: | ---: |
| 1' | Sun-centered circle | 0.051504 | 0.046848 | 0.042350 | 0.099385 |
| 1' | Free-center circle | 0.021718 | 0.017068 | 0.012948 | 0.049116 |
| 1' | Focus-constrained ellipse | 0.021759 | 0.017124 | 0.013161 | 0.048913 |
| 2' | Sun-centered circle | 0.051629 | 0.046963 | 0.042165 | 0.099916 |
| 2' | Free-center circle | 0.021879 | 0.017224 | 0.012521 | 0.048463 |
| 2' | Focus-constrained ellipse | 0.021920 | 0.017268 | 0.012740 | 0.048260 |
| 5' | Sun-centered circle | 0.052027 | 0.047300 | 0.043520 | 0.101501 |
| 5' | Free-center circle | 0.022431 | 0.017721 | 0.012080 | 0.046529 |
| 5' | Focus-constrained ellipse | 0.022475 | 0.017760 | 0.011934 | 0.046580 |



For the 1 arcminute data, the fitted parameters were:



- Sun-centered circle radius: 1.52619 AU

- Free-circle center: (-0.09834, 0.01994) AU

- Free-circle radius: 1.48445 AU

- Fitted ellipse semi-major axis: 1.48547 AU

- Fitted ellipse eccentricity: 0.06689

- Fitted ellipse orientation: -12.63 degrees



The hidden generating values were $a = 1.52366\,\mathrm{AU}$ and $e = 0.09341$, so the focus-constrained ellipse fit recovered the correct general scale and a nonzero eccentricity, although the reconstruction underestimates the eccentricity.

There is also a mathematical reason the free-center circle can mimic the fitted ellipse so closely. For small eccentricity,

$$
r(\theta)=\frac{p}{1+e\cos(\theta-\omega)}
\approx a-ae\cos(\theta-\omega),
$$

so the leading noncircular variation has amplitude approximately $ae$. A circle of radius $R$ whose center is displaced from the Sun by a small distance $d$ has, to first order,

$$
r(\theta)\approx R+d\cos(\theta-\theta_c).
$$

For the 1 arcminute fits,

$$
ae \approx 1.48547(0.06689)=0.09936\,\mathrm{AU},
$$

while the fitted free-circle center is displaced from the Sun by

$$
\sqrt{(-0.09834)^2+(0.01994)^2}\approx 0.10034\,\mathrm{AU}.
$$

These values are nearly identical. Thus the shifted circle reproduces the first-order radial variation of the small-eccentricity ellipse. Their geometric difference appears mainly at second order, on the rough scale $ae^2\approx 0.00665\,\mathrm{AU}$, which is smaller than the approximately $0.022\,\mathrm{AU}$ reconstruction/model residual. This helps explain why the two fitted curves are effectively indistinguishable in the present reconstruction.



### 1 arcminute model fits
![Model fits 1 arcmin](result%20images/model_fits_1arcmin.png)



### 2 arcminute model fits
![Model fits 2 arcmin](result%20images/model_fits_2arcmin.png)



### 5 arcminute model fits
![Model fits 5 arcmin](result%20images/model_fits_5arcmin.png)



The Sun-centered circular model is clearly worse. At 1 arcminute, changing from a Sun-centered circle to the ellipse lowers RMSE by about 57.75 percent.



However, the free-center circle and focus-constrained ellipse are almost indistinguishable for the reconstructed points. The free-center circle is slightly better in RMSE by only about 0.19 percent at every tested noise level. This difference is far smaller than the overall reconstruction residual and is not practically meaningful evidence that the physical orbit is circular. The small eccentricity, limited reconstructed phase coverage, and first-order equivalence between a shifted circle and a focus ellipse all make the two models difficult to distinguish here.



## Effect of observational noise
![Noise versus RMSE](result%20images/step7_noise_vs_rmse.png)



| Noise | Sun-circle RMSE | Free-circle RMSE | Ellipse RMSE | Ellipse minus free-circle RMSE | Ellipse vs free-circle difference | Ellipse improvement vs Sun circle |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 1' | 0.051504 | 0.021718 | 0.021759 | 0.000041 | 0.188% worse | 57.75% better |
| 2' | 0.051629 | 0.021879 | 0.021920 | 0.000041 | 0.188% worse | 57.54% better |
| 5' | 0.052027 | 0.022431 | 0.022475 | 0.000043 | 0.193% worse | 56.80% better |



Among the three tested noise levels, none makes the focus-constrained ellipse clearly outperform the free-center circle. Even at 1 arcminute, the two models are effectively tied. Therefore this experiment does not identify a numerical accuracy threshold at which the ellipse begins to win; it only shows that 1 arcminute is not sufficient for this particular reconstruction pipeline. The dominant limitations appear to be reconstruction error, limited reconstructed phase coverage, and the fact that a shifted circle approximates a low-eccentricity focus ellipse to first order. Testing noise below 1 arcminute, using a better-conditioned reconstruction, and repeating each noise level over multiple random realizations would be needed to estimate a meaningful threshold.



---
# 7. LLM Prompts and Main Suggestions
The LLM was used throughout the project as a collaborator. The prompts below are representative of the actual interaction; wording has been lightly cleaned for readability while preserving the original requests.



## Prompt 1: Build the synthetic observation model
> Start with the code for synthetic Tycho-like observations of Mars. Explain how the Uraniborg coordinates are used and what equations are needed to generate the synthetic data. Include Earth motion, Mars motion, Earth rotation, and observational noise.



### Main LLM suggestions
The LLM proposed:



- use a simplified heliocentric two-planet model with the Sun fixed at the origin;

- represent Earth and Mars using Kepler ellipses;

- solve Kepler's equation numerically with Newton's method;

- use Uraniborg latitude and longitude together with sidereal time to construct the local observer position and local east/north/up basis;

- convert ecliptic coordinates to equatorial coordinates using Earth's obliquity;

- calculate both RA/declination and altitude/azimuth;

- add angular noise in the tangent plane of the line-of-sight vector rather than directly perturbing Cartesian coordinates;

- reject observations when Mars is too low or the Sun is too high.



These suggestions became the basis of the synthetic data generator.



## Prompt 2: Verify the simulation independently
> Verify the simulation code independently. Check energy and angular momentum for the noiseless Earth and Mars orbits and give me a standalone verification program.



### Main LLM suggestions
The LLM recommended comparing the numerical values with the exact Kepler formulas



$$

\mathcal{E} = -\frac{\mu}{2a}

$$



$$

h = \sqrt{\mu a(1-e^2)}

$$



It also recommended using relative errors instead of deciding visually that a curve looked flat. This led to an independent verification file rather than reusing the same functions blindly inside the data generator.



## Prompt 3: Visualize Mars in Tycho's sky
> Plot the apparent position of Mars in Tycho's sky using the synthetic CSV files.



### Main LLM suggestions
The LLM recommended plotting both RA/declination and altitude/azimuth. It also warned against connecting points across the 0/360 degree wrap in RA or azimuth, so scatter plots were used.



## Prompt 4: Reconstruct Mars from lines of sight
> Convert the observations into lines of sight and reconstruct heliocentric Mars positions without using the hidden Mars orbital parameters.



### Main LLM suggestions
The first suggestion was a repeated-phase triangulation method based on pairs of rays. That initial method was not reliable and is discussed in the critique below. After the failure was detected, the LLM proposed a corrected multi-ray method:



- search a broad range of candidate orbital periods rather than inserting Mars's true period;

- group observations separated by approximately integer multiples of the candidate period;

- use three or more rays for each candidate heliocentric point;

- solve a least-squares closest-point problem for the full ray bundle;

- reject bundles with poor geometric baseline;

- validate the inferred radii and out-of-plane coordinate before accepting the reconstruction.



The corrected method inferred a period of 689.25 days and physically reasonable heliocentric Mars distances.



## Prompt 5: Compare circle and ellipse models
> Fit a Sun-centered circle, a circle with arbitrary center, and an ellipse to the reconstructed Mars positions. Compute comparable residual errors.



### Main LLM suggestions
The LLM suggested using a common geometric residual for the final comparison: the Euclidean distance from each reconstructed point to the nearest point on each fitted curve. This avoided comparing incompatible residual definitions across different model types.



## Prompt 6: Interpret the noise experiment
> Compare the 1, 2, and 5 arcminute results. How much observational accuracy is needed before the ellipse clearly wins?



### Main LLM suggestions
The LLM did not force the historical conclusion. It observed that the Sun-centered circle is clearly worse, but the free-center circle and focus-constrained ellipse are almost tied at every tested noise level. The final interpretation is that measurement noise is not the only limitation: reconstruction error, limited reconstructed phase coverage, and the small-eccentricity similarity between a shifted circle and a focus ellipse are all important.



## Prompt 7: Interpret the result as if reasoning from the data
> Interpret the numerical results. Did the AI reason from the observations, or did it merely recall that Mars follows an ellipse?



### Main LLM suggestion
The LLM concluded that the reconstructed-data comparison strongly disfavors a Sun-centered circle, but does not justify claiming that the focus-constrained ellipse clearly beats the shifted-circle alternative. This interpretation is important because it follows the computed residuals even though the LLM already knows the historical fact that Mars follows an ellipse.



---
# 8. Critique of the LLM: What Was Useful and What Was Not
The LLM was most useful as a source of mathematical structure, implementation ideas, and verification strategies. It was less reliable when a plausible geometric method had hidden conditioning problems.



## Useful contributions
### 1. It translated the physical setup into a modular computational model
The LLM separated the problem into orbital motion, coordinate transformations, Earth rotation, observer position, angular observations, noise, reconstruction, and model fitting. This modular organization made the program easier to inspect line by line and easier to verify.



### 2. It connected the code to explicit equations
Rather than only supplying Python code, the LLM explained Kepler's equation, the ellipse parameterization, velocity formulas, conserved energy, conserved angular momentum, RA/declination conversion, and the observer geometry. This made it possible to check whether individual code blocks had a mathematical interpretation.



### 3. Its verification strategy was strong
The most useful suggestion was to compare the computed energy and angular momentum with independent closed-form theoretical values. The approximately 1e-15 relative errors provided a much stronger verification than simply looking at orbital plots.



### 4. It helped identify sanity and conditioning checks

The project benefited from checks such as:

- Earth should remain near 1 AU in the known Earth model;
- recovered ray ranges should be positive;
- multi-ray systems should not be nearly singular;
- a planar reconstruction should have small out-of-plane coordinates when expressed in the chosen orbital frame;
- an inferred repetition period should be supported by the observation geometry rather than inserted as the known Mars period.

The hidden Mars distance range 1.38 to 1.67 AU is used only in Step 5-1 as a post-hoc ground-truth check. It is not a reconstruction criterion and does not feed into Steps 6 or 7.

### 5. It adapted when the numerical evidence contradicted expectations
The final model comparison did not show the ellipse clearly winning. The LLM correctly interpreted this instead of claiming that the result had confirmed Kepler simply because Mars is historically known to follow an ellipse.

### 6. Step 5-1 provided an independent downstream validation
The separate Step 5-1 notebook intentionally uses the hidden Mars parameters only after the blind reconstruction has already been saved by Step 4. It reads the completed 1 arcminute reconstruction and overlays it with the true synthetic orbit. Because Steps 6 and 7 read the Step 4 and Step 6 outputs directly rather than any Step 5-1 output, this check verifies the reconstruction without altering the reported fitting results.



## Weak or incorrect contributions
### 1. The first reconstruction method was wrong in an important way

The initial LLM suggestion used pairs of repeated-phase lines of sight and chose the candidate period that made the rays intersect most closely. That method produced reconstructed radii near 1 AU. The result looked geometrically clean, but the depth estimate was unstable because nearly repeated observer geometry can make two-ray triangulation poorly conditioned. A small intersection residual by itself does not imply that the recovered heliocentric distance is reliable.

The final method corrected this weakness by using multi-ray least squares rather than pairs alone. It also requires a minimum observer baseline, rejects bundles with $\operatorname{cond}(A)>10^{10}$, and requires all recovered ray ranges to be positive. Step 5-1 independently confirms that this corrected reconstruction agrees with the hidden synthetic orbit, but that check is downstream of the reconstruction and does not alter the reported results.

### 2. The LLM initially had a natural bias toward expecting the ellipse to win
Because an LLM already knows Kepler's laws, it is easy for it to frame the experiment around the expected historical answer. The actual fitted residuals did not support a strong ellipse-over-shifted-circle conclusion. The final interpretation had to remain disciplined and follow the computed values rather than the prior historical fact.



### 3. Some modeling choices were practical rather than uniquely correct
The 10-day sampling interval, fixed 22:00 UTC observation time, 10-degree Mars altitude cutoff, -6-degree Sun altitude cutoff, planar planetary orbits, and simplified sidereal-time calculation are reasonable choices for a synthetic experiment, but they are not unique or fully historical. These choices should be described as assumptions, not as exact features of Tycho Brahe's observing program.



### 4. The reconstruction does not recover the entire orbit equally well
Only a subset of orbital phases produced sufficiently good repeated multi-ray geometry. The reconstructed points therefore cover limited orbital phase rather than a uniformly sampled complete ellipse. This contributes to the near tie. In addition, for a low-eccentricity orbit a shifted circle already matches the leading first-order radial variation of a focus ellipse, so distinguishing the two requires reconstruction errors smaller than the second-order geometric difference.



## Overall assessment of the LLM
The LLM was useful when treated as a collaborator that generates equations, code structures, diagnostic ideas, and alternative approaches. It was not reliable enough to accept without independent mathematical and physical checks. The failed first reconstruction is a useful example: the code ran and the geometry looked plausible, but weak parallax made the recovered distance unstable. Examining the observer geometry and conditioning led to the corrected multi-ray method, while the separate Step 5-1 hidden-orbit comparison provided a downstream validation of the final reconstruction.



---
# 9. Did the AI Reason Like Kepler, or Did It Merely Remember Kepler?
The experiment does not justify claiming that the AI independently rediscovered Kepler's law in the historical sense. The LLM already knows that Mars follows an elliptical orbit, and the synthetic data themselves were generated from a hidden Kepler ellipse. Therefore the AI cannot be treated as an observer with no prior astronomical knowledge.



However, the final analysis did show a meaningful form of data-based reasoning. When the residuals were computed, the LLM did not simply say that the ellipse must be best because Kepler discovered ellipses. The numerical evidence showed that the Sun-centered circle was clearly worse, while the free-center circle and focus-constrained ellipse had almost identical residual errors. In fact, the free-center circle had a slightly lower RMSE at all three tested noise levels. The LLM therefore concluded only what this reconstruction supports: a Sun-centered circle is strongly disfavored, but the available reconstructed points are not sufficient to distinguish the focus-constrained ellipse clearly from a shifted circle.



The best description is that the AI acted more like a modern scientific collaborator than a true historical Kepler. It supplied hypotheses and computational methods, and one early pairwise reconstruction method had a conditioning problem that was corrected in the final multi-ray implementation. The separate Step 5-1 ground-truth check then confirmed that the corrected reconstruction was physically consistent, without entering the later fitting calculations. The AI's strongest reasoning occurred when it followed the numerical evidence even when that evidence did not reproduce the expected historical conclusion. The project therefore demonstrates both the usefulness and the limitation of LLM-assisted scientific work: the LLM can accelerate exploration, but the student must still verify the mathematics and decide what the data actually prove.



---
# Discussion: Kepler's Problem as Model Fitting from Data
This problem is an early example of model fitting from data because we begin with observations and ask which mathematical model best explains them. Instead of assuming the correct orbit in advance, we compare candidate models, fit their parameters, and use residual errors to determine how well each model explains the reconstructed observations.



Kepler did something conceptually similar when he analyzed Tycho Brahe's measurements of Mars. He did not simply accept the traditional circular description because it was mathematically attractive. The observational discrepancies forced him to consider a different model. In that sense, the historical process resembles modern approximation: observations constrain a family of candidate mathematical descriptions, and the model is judged by how well it accounts for the data.



The general workflow is:



1. collect observations;

2. choose candidate models;

3. fit model parameters;

4. compute residual errors;

5. compare competing explanations;

6. retain only conclusions justified by the data.



This is conceptually related to modern machine learning and approximation theory. In many modern problems, the true input-output relationship is unknown. A parameterized model is fitted to observations by minimizing a loss or error. Kepler was not using a neural network or gradient descent, but the underlying scientific logic is similar: infer mathematical structure from data rather than imposing the preferred model in advance.



The current experiment also demonstrates an important limitation of model fitting. The fitted reconstructed points strongly disfavor the simplest Sun-centered circle, but they do not clearly distinguish the focus-constrained ellipse from a shifted circle. A scientifically valid conclusion must therefore stop at what the reconstruction and residuals support. Knowing the correct historical answer is not a substitute for evidence from the experiment.



---
# Files Submitted with This Report
## Code
- `codes/synthetic_data_generation(step_1).ipynb`

- `codes/verify_orbits(step_2).ipynb`

- `codes/plot_mars_sky(step_3).ipynb`

- `codes/reconstruct_mars(step_4).ipynb`

- `codes/plot_reconstructed_mars(step_5).ipynb`

- `codes/rigorous_reconstructed_mars(step_5-1).ipynb`

- `codes/fit_models(step6).ipynb`

- `codes/noise_comparison(step7).ipynb`

- `codes/LLM_reasoning(step8).ipynb`

- `codes/dicussion_model_fitting(step9).ipynb`



## Data
- `data/synthetic_mars_1arcmin.csv`

- `data/synthetic_mars_2arcmin.csv`

- `data/synthetic_mars_5arcmin.csv`

- `data/orbit_verification_summary.csv`

- `data/earth_orbit_verification.csv`

- `data/mars_orbit_verification.csv`

- `data/period_search_1arcmin.csv`

- `data/period_search_2arcmin.csv`

- `data/period_search_5arcmin.csv`

- `data/mars_reconstructed_1arcmin.csv`

- `data/mars_reconstructed_2arcmin.csv`

- `data/mars_reconstructed_5arcmin.csv`

- `data/model_fit_summary.csv`

- `data/step7_noise_comparison.csv`



## Figures
All generated figures are stored in the `result images` directory.
